# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring**

The starter pipeline already demonstrates this lane end-to-end: a transparent baseline rule, three learned models (LR, DT, RF), client-holdout validation, and a ranked review queue with reason codes. On the 30k-row anonymized slice, the random forest lifts Precision@50 from ~0.24 (baseline) to ~0.74 — a 3× improvement. The data has strong, observable signals for this task: trend direction, impression volume, content age, freshness, word count, position, CTR, and engagement rates. The output (a prioritized queue with action tags) is directly usable by content editors. This lane also extends naturally to the full warehouse (79M daily rows) where I can define stronger future-window labels and test richer feature windows.

In [ ]:
# Lane 2 is the Refresh / Content Opportunity Scoring lane
# This cell intentionally left as a marker — the lane choice is documented in markdown above

## 2. The question: decision, action, cost of a wrong call

**What decision does this improve?**
Which content pages should an editor review first for potential refresh, expansion, protection, pruning, or monitoring — given limited editorial capacity.

**Who acts on the output, and what do they do?**
A content editor or SEO strategist receives a ranked queue of pages with a score, a suggested action (refresh / expand / protect / prune / monitor), and a reason code (e.g., `stale_visible_page`, `declining_with_demand`, `thin_visible_page`). They open the top-K pages, assess manually, and decide on an edit action.

**What does a wrong answer cost?**
- False positive (high score, page doesn't need action): wasted editor time reviewing a healthy page — opportunity cost of ~15–30 minutes per page.
- False negative (low score, page actually declining): missed decline that continues losing traffic — cost compounds over weeks/months.
- The baseline rule already catches ~12/50 top pages (Precision@50 = 0.24). A learned model catching ~37/50 (Precision@50 = 0.74) means ~25 more correct priorities per 50 reviewed — directly translating to editor efficiency.

**Why does data/ML help at all?**
A single if-rule (e.g., "pages not updated in 180 days with >500 impressions") captures one signal. But real decline risk is multi-factorial: a fresh page with dropping CTR, an old page with stable position but falling engagement, a high-volume page with rising AI-referral cannibalization. These signals interact non-linearly and shift over time. A learned ranking combines dozens of observed signals, weights them by actual predictive power on held-out clients, and adapts when retrained — something a static rule cannot do.

In [ ]:
# Framing answers documented in markdown above — this cell is a placeholder for any exploratory code

## 3. Quick look at the data (2-3 real numbers)

```python
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('Shape:', df.shape)
print('Clients:', df['client_id'].nunique())
print('Label balance (trend_direction == down):', (df['trend_direction'] == 'down').mean())
print()

# Key signal: stale visible pages
stale_visible = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
print(f'Stale visible pages (days_since_last_update >= 180, impressions_90d >= 500): {len(stale_visible)} ({len(stale_visible)/len(df)*100:.1f}%)')
print(f'  Of those, declining: {(stale_visible["trend_direction"] == "down").mean()*100:.1f}%')
print()

# Key signal: declining with demand
declining_demand = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]
print(f'Declining with demand (trend=down, impressions_90d >= 100): {len(declining_demand)} ({len(declining_demand)/len(df)*100:.1f}%)')
print()

# Key signal: thin visible pages
thin_visible = df[(df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_90d'] >= 250)]
print(f'Thin visible pages (word_count 1-1199, impressions_90d >= 250): {len(thin_visible)} ({len(thin_visible)/len(df)*100:.1f}%)')
print(f'  Of those, declining: {(thin_visible["trend_direction"] == "down").mean()*100:.1f}%')
```

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('Shape:', df.shape)
print('Clients:', df['client_id'].nunique())
print('Label balance (trend_direction == down):', (df['trend_direction'] == 'down').mean())
print()

# Key signal: stale visible pages
stale_visible = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
print(f'Stale visible pages (days_since_last_update >= 180, impressions_90d >= 500): {len(stale_visible)} ({len(stale_visible)/len(df)*100:.1f}%)')
print(f'  Of those, declining: {(stale_visible["trend_direction"] == "down").mean()*100:.1f}%')
print()

# Key signal: declining with demand
declining_demand = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]
print(f'Declining with demand (trend=down, impressions_90d >= 100): {len(declining_demand)} ({len(declining_demand)/len(df)*100:.1f}%)')
print()

# Key signal: thin visible pages
thin_visible = df[(df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_90d'] >= 250)]
print(f'Thin visible pages (word_count 1-1199, impressions_90d >= 250): {len(thin_visible)} ({len(thin_visible)/len(df)*100:.1f}%)')
print(f'  Of those, declining: {(thin_visible["trend_direction"] == "down").mean()*100:.1f}%')
print()

# Additional signal: page one decay risk
page_one_decay = df[(df['avg_position'] > 0) & (df['avg_position'] <= 10) & (df['content_age_days'] >= 180)]
print(f'Page one decay risk (avg_position 1-10, content_age_days >= 180): {len(page_one_decay)} ({len(page_one_decay)/len(df)*100:.1f}%)')
print(f'  Of those, declining: {(page_one_decay["trend_direction"] == "down").mean()*100:.1f}%')
print()

# Additional signal: low CTR visible pages
low_ctr_visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)]
print(f'Low CTR visible pages (impressions_90d >= 500, position 1-20, ctr < 0.5): {len(low_ctr_visible)} ({len(low_ctr_visible)/len(df)*100:.1f}%)')
print(f'  Of those, declining: {(low_ctr_visible["trend_direction"] == "down").mean()*100:.1f}%')

## 4. Careful words: what I can and can't claim

**What this work WILL be able to say (honest claims):**

- **Observed**: "Pages with signal combination X had a Y% rate of declining trend in the held-out client test set."
- **Measured**: "The learned ranking achieves Precision@50 = Z on client-holdout validation, vs. 0.24 for the baseline rule."
- **Directional**: "Signal A shows a stronger association with decline than Signal B, controlling for volume."
- **Decision-support**: "Reviewing the top-50 pages from this ranking would catch ~37 declining pages vs. ~12 from the baseline rule — helping editors prioritize limited review capacity."
- **Scope-limited**: "Results are on a 30k-row anonymized slice (32 clients, 90-day window). The full warehouse (79M daily rows) requires re-validation."

**What this work will NEVER claim:**

- ❌ "This model predicts Google's algorithm."
- ❌ "Refreshing a page from this queue will cause traffic recovery." (Causal claim — requires experiment, not observational data)
- ❌ "This identifies the exact reason a page declined." (We observe correlates, not causes)
- ❌ "The model generalizes to all websites / all time periods." (Only tested on this pseudonymized slice)
- ❌ "The score is a probability of future decline." (It's a ranking score calibrated on a proxy label from the *same* window)

**Label honesty note**: The starter label `is_declining_label = (trend_direction == "down")` is a **proxy** — it measures decline *within the current 90-day window*, not a future outcome. A stronger capstone will define a forward-window label (e.g., features from prior 90 days → decline in next 30 days) with strict temporal separation. Until then, all claims are about *ranking quality on the observed proxy*, not future prediction.

In [ ]:
# Claims documented in markdown above — no code needed here

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.